# Campaign Performance & Classification Analysis

Analysis of campaign acceptance rates, target distribution, and response rates by customer segment.

In [1]:
# Imports and load data
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('../data/preprocessed/marketing_campaign_preprocessed.csv')

## 1. Campaign Acceptance Rates

In [2]:
cmp_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']

print('=== CAMPAIGN ACCEPTANCE RATES ===')
for col in cmp_cols:
    accepted = df[col].sum()
    total = len(df)
    print(f'{col}: {accepted}/{total} ({accepted/total*100:.1f}%)')

=== CAMPAIGN ACCEPTANCE RATES ===
AcceptedCmp1: 144/2229 (6.5%)
AcceptedCmp2: 30/2229 (1.3%)
AcceptedCmp3: 163/2229 (7.3%)
AcceptedCmp4: 167/2229 (7.5%)
AcceptedCmp5: 162/2229 (7.3%)
Response: 334/2229 (15.0%)


## 2. Response (Target) Distribution

In [3]:
print('=== RESPONSE (TARGET) DISTRIBUTION ===')
print(df['Response'].value_counts())
print()
print(f"Class balance: {df['Response'].mean()*100:.1f}% accepted, {(1-df['Response'].mean())*100:.1f}% rejected")

=== RESPONSE (TARGET) DISTRIBUTION ===
Response
0    1895
1     334
Name: count, dtype: int64

Class balance: 15.0% accepted, 85.0% rejected


## 3. Customers Who Accepted Multiple Campaigns

In [4]:
df['Total_Accepted'] = df[cmp_cols].sum(axis=1)

print('=== CUSTOMERS WHO ACCEPTED MULTIPLE CAMPAIGNS ===')
print(df['Total_Accepted'].value_counts().sort_index())
print()
print(f"Customers who accepted ZERO campaigns: {(df['Total_Accepted']==0).sum()} ({(df['Total_Accepted']==0).mean()*100:.1f}%)")
print(f"Customers who accepted 1+ campaigns: {(df['Total_Accepted']>=1).sum()} ({(df['Total_Accepted']>=1).mean()*100:.1f}%)")

=== CUSTOMERS WHO ACCEPTED MULTIPLE CAMPAIGNS ===
Total_Accepted
0    1621
1     369
2     142
3      51
4      36
5      10
Name: count, dtype: int64

Customers who accepted ZERO campaigns: 1621 (72.7%)
Customers who accepted 1+ campaigns: 608 (27.3%)


## 4. Cross-Tab: Response vs Past Campaign Acceptance

In [5]:
df['Accepted_Any_Past'] = df[['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']].max(axis=1)

print('=== CROSS-TAB: RESPONSE vs PAST CAMPAIGN ACCEPTANCE ===')
print(pd.crosstab(df['Accepted_Any_Past'], df['Response'], margins=True, normalize='index').round(3) * 100)

=== CROSS-TAB: RESPONSE vs PAST CAMPAIGN ACCEPTANCE ===
Response              0     1
Accepted_Any_Past            
0                  91.7   8.3
1                  59.3  40.7
All                85.0  15.0


## 5. Response Rate by Cluster (K-Means k=4)

In [6]:
df_encoded = pd.get_dummies(df, columns=['Education', 'Living_With'], drop_first=True)

numeric_features = df_encoded.select_dtypes(include=['int64', 'float64', 'uint8']).columns.tolist()
exclude = ['Response', 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Total_Accepted', 'Accepted_Any_Past']
numeric_features = [c for c in numeric_features if c not in exclude]

X = StandardScaler().fit_transform(df_encoded[numeric_features])
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

print('=== RESPONSE RATE BY CLUSTER ===')
print(df.groupby('Cluster').agg(
    count=('Response', 'count'),
    response_rate=('Response', 'mean'),
    avg_income=('Income', 'mean'),
    avg_spend=('Total_Spending', 'mean')
).round(3))

=== RESPONSE RATE BY CLUSTER ===
         count  response_rate  avg_income  avg_spend
Cluster                                             
0          611          0.141   54172.343     20.416
1          700          0.239   73184.441     27.747
2          898          0.087   33204.918     11.188
3           20          0.150   45672.400     16.451
